# OpenEnv Sudoku with GRPO using TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/trl/blob/main/examples/grpo_sudoku/grpo_sudoku.ipynb)

![trl banner](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)


With [**Transformers Reinforcement Learning (TRL)**](https://github.com/huggingface/trl), you can train a model that learns to **play Sudoku** through interaction and reinforcement.

This notebook covers the GRPO step. It starts from [`openenv/Qwen3-1.7B-Sudoku-SFT`](https://huggingface.co/openenv/Qwen3-1.7B-Sudoku-SFT), a Qwen3-1.7B that already learned to read the grid and fill it with a short SFT warmup. To build that model yourself (the data and the SFT step), follow the [OpenEnv Sudoku tutorial](https://huggingface.co/docs/openenv/tutorials/sudoku-grpo).

- [TRL GitHub Repository](https://github.com/huggingface/trl) -- star us to support the project!
- [Official TRL Examples](https://huggingface.co/docs/trl/example_overview)
- [Community Tutorials](https://huggingface.co/docs/trl/community_tutorials)
- [OpenEnv](https://github.com/openenv/OpenEnv)


An **agentic environment** is a setting where a model can take actions, observe outcomes, and adjust its behavior based on feedback, similar to how humans learn from trial and error.
In this case, the agent interacts with the **Sudoku** environment through the [**OpenEnv**](https://github.com/openenv/OpenEnv) framework, which standardizes multi-agent and RL-style text environments.

In [Sudoku](https://en.wikipedia.org/wiki/Sudoku), every empty cell of a 9x9 grid gets a digit from 1 to 9 so that each row, each column and each 3x3 box contains every digit once. The model fills one cell per call to a `place(row, col, number)` tool and gets the updated grid back. The environment rejects any digit that is not the solution of its cell, and two rejected moves in a row end the game.


We'll fine-tune a model using **GRPO** (Group Relative Policy Optimization) via TRL.
Using `environment_factory`, the trainer automatically handles:
1. Creating environment instances for each rollout.
2. Generating model completions and parsing tool calls.
3. Stepping through the environment with the model's actions.
4. Collecting rewards and managing the interaction loop.

This means you only need to define the environment class and reward function -- the trainer takes care of the rest.

The SFT warmup comes first because the base model solves no puzzle, which leaves GRPO little to learn from. In our run, 75 GRPO steps took about 15 hours on one A100 80GB, and took the model from 10 to 22 solved puzzles out of 30 at 26 empty cells.


## Install dependencies

We'll start by installing **TRL** (with vLLM support), the **OpenEnv** TextArena environment, and **trackio** for logging.

In [ ]:
!pip install -Uq "trl[vllm]" "openenv-textarena @ git+https://github.com/openenv/OpenEnv.git@v0.10.0#subdirectory=envs/textarena_env" trackio

### Log in to Hugging Face

Log in to your **Hugging Face** account to save your fine-tuned model, track your experiment results directly on the Hub or access gated models. You can find your **access token** on your [account settings page](https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Define the prompt

This prompt gives the model the rules and tells it to use the `place` tool. The `environment_factory` pattern uses tool calling to interact with the environment, so the model needs to know which tool to call. The starting grid is appended to it by `reset()`.

In [ ]:
prompt = """You are playing Sudoku. Fill every empty cell (shown as '.') of the 9x9 grid with a digit from 1 to 9 so that \
each row, each column and each 3x3 box contains every digit exactly once.

Use the tool `place` to fill one cell at a time. Rows and columns are numbered from 1 to 9. A digit that is not the \
solution of that cell is a mistake. Two mistakes in a row end the game.

"""

## Define the environment

The `SudokuEnv` class wraps the OpenEnv TextArena Sudoku environment into the interface expected by `environment_factory`.

When you pass `environment_factory=SudokuEnv` to the trainer, it will:
1. Create a new `SudokuEnv()` instance for each rollout episode.
2. Call `reset()` to start a new game (returns the initial observation or `None`).
3. Automatically generate model completions, parse tool calls, and invoke the corresponding methods (e.g., `place(...)`).
4. Repeat until the environment signals `done=True` or the max completion length is reached.

The environment exposes its public methods as tools. Any public method (other than `reset`) with a docstring is automatically discovered and exposed as a callable tool. Here, the `place` method lets the model fill a cell and receive the updated grid.

Each training step plays 64 games, one environment session each. [Duplicate the `openenv/sudoku` Space](https://huggingface.co/spaces/openenv/sudoku?duplicate=true) and set these variables on your copy, then use its URL below:
- `TEXTARENA_KW_CLUES=55`: about 26 empty cells (the Space default is 60 clues, about 21).
- `MAX_CONCURRENT_ENVS=96`: 64 games, plus room for the sessions being closed between games.

`reset()` opens a new connection for every game. A Space can close WebSockets after a couple of hours, and the client then reconnects to a new session with a different board, in the middle of a game.

The class also tracks what the reward needs: how many cells were empty, how many the model filled and whether it solved the puzzle.

For more information, refer to the [TRL-OpenEnv documentation](https://huggingface.co/docs/trl/main/en/openenv).

In [ ]:
from huggingface_hub import get_token
from textarena_env import TextArenaAction, TextArenaEnv

env_url = "https://<user>-sudoku.hf.space"  # your copy of openenv/sudoku
token = get_token()
headers = {"Authorization": f"Bearer {token}"} if token else None


class SudokuEnv:
    def __init__(self):
        self.client = TextArenaEnv(base_url=env_url, headers=headers).sync()

    def reset(self, **kwargs) -> None | str:
        # One connection per game, so the Space's periodic WebSocket drops don't land mid-game, see
        # https://huggingface.co/docs/trl/openenv#long-runs-against-a-space
        self.client.close()
        self.client = TextArenaEnv(base_url=env_url, headers=headers).sync()
        result = self.client.reset()
        # The game returns the whole log each turn, so we store the previous full response and slice out only the
        # newly appended part.
        self._last_full = result.observation.messages[0].content
        board = self._last_full.split("Game Board:\n")[-1]
        self.empty = board.count(".")
        self.filled = 0
        self.solved = False
        self.done = False
        return board

    def place(self, row: int, col: int, number: int) -> str:
        """
        Place a number in an empty cell of the Sudoku grid.

        Args:
            row: Row number (1-9).
            col: Column number (1-9).
            number: Number to place (1-9).

        Returns:
            The updated grid, or the reason the move was rejected.
        """
        if self.done:
            raise ValueError("Game over.")
        result = self.client.step(TextArenaAction(message=f"[{row} {col} {number}]"))
        full = result.observation.messages[0].content
        feedback, self._last_full = full[len(self._last_full) :], full
        self.done = result.done
        if "invalid" in feedback.lower():
            return feedback.split("[GAME]")[-1].strip()
        self.filled += 1
        self.solved = result.done and result.reward >= 1.0
        return feedback.split("Board state: \n")[-1]

## Define the reward functions

Two rewards: 1 if the puzzle is solved, plus the fraction of empty cells filled. The SFT model solves only a few puzzles at 26 empty cells, so with the binary reward alone many groups would score all zeros and give no signal. The fraction filled separates games that stop at 5 cells from games that stop at 20. TRL logs each one separately, so `rewards/reward_solved/mean` tracks the share of solved puzzles.

In [ ]:
def reward_solved(environments, **kwargs) -> list[float]:
    return [1.0 if env.solved else 0.0 for env in environments]


def reward_progress(environments, **kwargs) -> list[float]:
    return [env.filled / env.empty for env in environments]

## Create the dataset

We create a dataset with repeated prompts to control the number of training episodes.
Each entry triggers one rollout episode during training. The prompt is formatted as a chat message.

In [ ]:
from datasets import Dataset

# 75 steps x 4 prompts
dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": prompt}]] * 300})

## Set GRPO Config

Next, we define the **GRPOConfig**, which controls all key training parameters.
Each step plays 4 prompts x 16 games. Thinking stays on (`enable_thinking=True`). Every move returns the whole grid, so a game at 26 empty cells can reach 16k tokens: `max_completion_length=16384` and one game per forward pass (`per_device_train_batch_size=1`), since 8 at once ran out of memory on an A100 80GB. `vllm_importance_sampling_mode="token_truncate"` clips per-token ratios instead of masking whole multi-turn sequences.

Steps get slower as the model gets better, because solved games are longer: from about 8 minutes to about 16 per step in our run.

In [ ]:
from trl import GRPOConfig

model_name = "openenv/Qwen3-1.7B-Sudoku-SFT"
output_dir = "Qwen3-1.7B-Sudoku-SFT-GRPO"

grpo_config = GRPOConfig(
    # Training schedule / optimization
    max_steps=75,
    learning_rate=1e-6,
    lr_scheduler_type="constant",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=64,
    bf16=True,

    # GRPO configuration
    num_generations=16,
    max_completion_length=16384,
    mask_truncated_completions=True,
    max_tool_calling_iterations=40,
    vllm_importance_sampling_mode="token_truncate",
    log_completions=True,
    num_completions_to_print=1,
    chat_template_kwargs={"enable_thinking": True},

    # vLLM configuration
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.35,

    # Logging / reporting
    output_dir=output_dir,
    report_to="trackio",
    logging_steps=1,
    save_steps=25,
    save_total_limit=1,

    gradient_checkpointing=True,

    # Hub integration
    push_to_hub=True,
)

## Create the `GRPOTrainer` and start training

Now we initialize the `GRPOTrainer` with `environment_factory=SudokuEnv`.

This tells the trainer to automatically handle the entire interaction loop:
- It creates a `SudokuEnv` instance for each episode.
- It generates model completions, parses tool calls (like `place`), and steps through the environment.
- It collects rewards and manages the `tool_mask` (which tokens are model-generated vs environment-generated) automatically.

No need to write a custom `rollout_func` or manage tokenization manually.

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    reward_funcs=[reward_solved, reward_progress],
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=SudokuEnv,
)

Show memory stats before training

In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

And train!

In [ ]:
trainer_stats = trainer.train()

Show memory stats after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

## Save and push to Hub

In [ ]:
trainer.save_model(output_dir)
trainer.push_to_hub()

## Load the fine-tuned model and run inference

Now let's test our fine-tuned model by loading it and playing a game of Sudoku.
We use the same `SudokuEnv` class to interact with the environment, and generate model responses with standard Transformers inference.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openenv/Qwen3-1.7B-Sudoku-SFT-GRPO"  # Replace with your HF username or organization

fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import json
import re

from transformers.utils import get_json_schema

TOOL_CALL = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)
tools = [get_json_schema(SudokuEnv.place)]


def play_sudoku(model, tokenizer):
    env = SudokuEnv()
    # Same as training: the first observation is appended to the prompt.
    messages = [{"role": "user", "content": prompt + env.reset()}]
    board = messages[0]["content"]

    while not env.done:
        prompt_text = tokenizer.apply_chat_template(
            messages, tools=tools, add_generation_prompt=True, tokenize=False, enable_thinking=True
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=512, do_sample=False)
        reply = tokenizer.decode(generated_ids[0][len(model_inputs.input_ids[0]):], skip_special_tokens=True)
        call = TOOL_CALL.search(reply)
        if call is None:
            print(reply)
            break
        arguments = json.loads(call.group(1))["arguments"]
        feedback = env.place(**arguments)
        rejected = "invalid" in feedback.lower()
        if not rejected:
            board = feedback
        print(f"{arguments}: {'rejected' if rejected else 'ok'}")
        messages += [
            {"role": "assistant", "content": reply},
            {"role": "tool", "name": "place", "content": feedback},
        ]

    print(board)
    print("Solved!" if env.solved else f"Filled {env.filled}/{env.empty} cells.")

Let's play the game!

In [ ]:
play_sudoku(fine_tuned_model, tokenizer)